# NLA Full pipeline on Colab

One-click pipeline that trains the reconstructor + verbalizer, evaluates the NLA, runs the paraphrase faithfulness test, and generates plots. All outputs go to Google Drive so nothing is lost on disconnect.

Set Runtime -> Change runtime type -> T4 GPU before Run all. Total time about 2 to 2.5 hours.

In [ ]:
!nvidia-smi

## 1. Get the latest code

In [ ]:
REPO_URL = "https://github.com/mohamedibrahim26/nla-kth.git"
import os
repo_dir = "/content/" + REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if os.path.exists(repo_dir):
    !cd $repo_dir && git pull
else:
    !git clone $REPO_URL
%cd $repo_dir
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao || true

## 2. Mount Drive (data + checkpoints persist here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nla/data'
assert os.path.exists(os.path.join(DATA_DIR,'summaries.jsonl')), 'Run notebooks 1 and 2 first!'
print('Drive data:', os.listdir(DATA_DIR))

## 3. Train reconstructor (~25-40 min)

In [ ]:
%cd /content/nla-kth
!python src/train_reconstructor.py --data_dir "$DATA_DIR" --output_dir "$DATA_DIR" --mode lora --epochs 5

## 4. Train verbalizer (~35-55 min)

In [ ]:
%cd /content/nla-kth
!python src/train_verbalizer.py --data_dir "$DATA_DIR" --output_dir "$DATA_DIR" --epochs 2

## 5. End-to-end evaluation (oracle / generated / random; ~15 min)

In [ ]:
%cd /content/nla-kth
!python src/evaluate_nla.py --data_dir "$DATA_DIR" --output_dir "$DATA_DIR"

## 6. Paraphrase faithfulness test (~45 min)

In [ ]:
%cd /content/nla-kth
!python src/paraphrase_test.py --data_dir "$DATA_DIR" --output_dir "$DATA_DIR"

## 7. Plots

In [ ]:
%cd /content/nla-kth
!python src/plot_results.py --data_dir "$DATA_DIR" --output_dir "$DATA_DIR/plots"
from IPython.display import Image, display
display(Image(filename=os.path.join(DATA_DIR, 'plots', 'fve_bar.png')))
display(Image(filename=os.path.join(DATA_DIR, 'plots', 'reconstructor_curve.png')))